# XGBoost con validacion LOSO y etiquetas individuales

Este notebook implementa el **Experimento A** con `XGBClassifier`: features derivadas de HR/R-R, ECG y variables temporales.

El objetivo es evaluar la generalizacion mediante Leave-One-Subject-Out Cross-Validation (LOSO). Las categorias `Low`, `Medium` y `High` se calculan individualmente para cada trabajador usando sus propios percentiles P33 y P66 de `FatigueIndex`.

## Definicion individual del target

Para cada trabajador se calculan sus propios P33 y P66 usando `FatigueIndex`. Esos limites generan sus etiquetas `Low`, `Medium` y `High` y se conservan durante todos los folds LOSO.

Las entradas son las columnas `_z`, ya normalizadas respecto a la baseline individual de cada trabajador. No se usa `FatigueIndex` como feature.

In [ ]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType as OnnxFloatTensorType

current_dir = Path.cwd()
candidate_roots = [
    current_dir,
    current_dir.parent,
    current_dir.parent.parent,
    current_dir.parent.parent.parent,
    current_dir / 'new',
]
ROOT_DIR = next(
    (
        candidate
        for candidate in candidate_roots
        if (candidate / 'W00' / 'PROCESSED' / 'combined_features_1min.csv').exists()
    ),
    current_dir,
)
DATA_DIR = ROOT_DIR
BASELINE_WINDOW_COUNT = 15
TARGET_COLUMN_CANDIDATES = ['FatigueIndex', 'fatigue_index']
LABELS = ['Low', 'Medium', 'High']
XGB_PARAM_GRID = {
    'classifier__n_estimators': [100, 200, 300, 500],
    'classifier__max_depth': [3, 5, 7, 9],
    'classifier__learning_rate': [0.1, 0.2, 0.5, 0.8],
}

print(f'Directorio de datos: {DATA_DIR}')

## Carga de datos

Cada CSV corresponde a un trabajador. El nombre de la carpeta (`W00`, ..., `W09`) se conserva como identificador para construir los folds. Durante la carga, cada trabajador recibe sus propios P33/P66 y su columna `FatigueLevel`; ninguna de estas columnas entra como feature del modelo.

In [2]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(
        fatigue_values,
        bins=[-np.inf, p33, p66, np.inf],
        labels=LABELS,
        right=False,
    ).astype(object)


worker_files = {
    worker_dir.name: worker_dir / 'PROCESSED' / 'combined_features_1min.csv'
    for worker_dir in sorted(DATA_DIR.glob('W[0-9][0-9]'))
    if (worker_dir / 'PROCESSED' / 'combined_features_1min.csv').exists()
}
if len(worker_files) < 2:
    raise ValueError(f'Se necesitan al menos 2 trabajadores procesados y se encontraron {len(worker_files)}: {list(worker_files)}')

data = {}
for worker, path in worker_files.items():
    frame = pd.read_csv(path)
    target_matches = [column for column in TARGET_COLUMN_CANDIDATES if column in frame.columns]
    if len(target_matches) != 1:
        raise ValueError(f'{worker}: no se encontro exactamente una columna FatigueIndex/fatigue_index')
    frame = frame.rename(columns={target_matches[0]: 'FatigueIndex'})
    frame['Trabajador'] = worker
    worker_fatigue = pd.to_numeric(frame['FatigueIndex'], errors='coerce')
    if worker_fatigue.dropna().empty:
        raise ValueError(f'{worker}: no contiene valores validos de FatigueIndex')
    worker_p33, worker_p66 = np.percentile(worker_fatigue.dropna(), [33, 66])
    frame['P33_worker'] = worker_p33
    frame['P66_worker'] = worker_p66
    frame['FatigueLevel'] = make_labels(worker_fatigue, worker_p33, worker_p66)
    data[worker] = frame

feature_columns = [
    'ECG_energy_z',
    'ECG_mean_z',
    'ECG_missing_peaks_z',
    'ECG_range_z',
    'ECG_samp_ent_z',
    'ECG_std_z',
    'HR_max_z',
    'HR_mean_z',
    'HR_min_z',
    'RMSSD_z',
    'SDNN_z',
    'pNN50_z',
    'HR_mean_ultimos_5min_z',
    'cambio_HR_vs_baseline_z',
    'tendencia_HR_z',
    'cambio_RMSSD_z',
]
missing_features = [
    column for column in feature_columns
    if column not in data[next(iter(data))].columns
]
if missing_features:
    raise ValueError(f'Faltan features requeridas: {missing_features}')
if not feature_columns:
    raise ValueError('No se encontraron columnas de entrada terminadas en _z')

print('Trabajadores:', list(data))
print('Features del Experimento A:', feature_columns)
print('Ventanas:', {worker: len(frame) for worker, frame in data.items()})
print('P33/P66 individuales:', {worker: (frame['P33_worker'].iloc[0], frame['P66_worker'].iloc[0]) for worker, frame in data.items()})

Trabajadores: ['W00', 'W01', 'W02', 'W03', 'W04', 'W05', 'W06', 'W07', 'W08', 'W09']
Features del Experimento A: ['ECG_energy_z', 'ECG_mean_z', 'ECG_missing_peaks_z', 'ECG_range_z', 'ECG_samp_ent_z', 'ECG_std_z', 'HR_max_z', 'HR_mean_z', 'HR_min_z', 'RMSSD_z', 'SDNN_z', 'pNN50_z', 'HR_mean_ultimos_5min_z', 'cambio_HR_vs_baseline_z', 'tendencia_HR_z', 'cambio_RMSSD_z']
Ventanas: {'W00': 104, 'W01': 637, 'W02': 603, 'W03': 407, 'W04': 562, 'W05': 74, 'W06': 35, 'W07': 67, 'W08': 133, 'W09': 16}
P33/P66 individuales: {'W00': (np.float64(-4.910791407727056), np.float64(-2.404562131467878)), 'W01': (np.float64(0.19857741939524748), np.float64(1.9952807675528523)), 'W02': (np.float64(0.15869861286917653), np.float64(1.8951128189329092)), 'W03': (np.float64(2.64758366326375), np.float64(6.383633710342336)), 'W04': (np.float64(-0.665956528377168), np.float64(-0.26155828390214836)), 'W05': (np.float64(-0.18718743181786043), np.float64(1.001663500738519)), 'W06': (np.float64(-2.177750214358166),

## Etiquetas y evaluacion

Las etiquetas se calculan una vez por trabajador con sus propios percentiles P33 y P66. En cada fold LOSO se dejan fuera todas las ventanas del trabajador TEST, pero sus limites individuales solo se usan para interpretar sus etiquetas; no se incorporan como features del modelo.

La matriz de confusion utiliza siempre el orden `Low`, `Medium`, `High`.

In [ ]:
def make_labels(fatigue_values, p33, p66):
    return pd.cut(
        fatigue_values,
        bins=[-np.inf, p33, p66, np.inf],
        labels=LABELS,
        right=False,
    ).astype(object)


def get_worker_percentiles(worker_frame):
    worker_fatigue = pd.to_numeric(worker_frame['FatigueIndex'], errors='coerce').dropna()
    if worker_fatigue.empty:
        raise ValueError(f"{worker_frame['Trabajador'].iloc[0]} no contiene valores validos de FatigueIndex")
    return np.percentile(worker_fatigue, [33, 66])


def build_model(n_estimators=100, max_depth=3, learning_rate=0.1):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', XGBClassifier(
            objective='multi:softprob',
            num_class=len(LABELS),
            eval_metric='mlogloss',
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            reg_lambda=1.0,
            tree_method='hist',
            n_jobs=1,
            random_state=42,
        )),
    ])


def encode_labels(labels):
    label_codes = pd.Categorical(labels, categories=LABELS).codes
    if (label_codes < 0).any():
        raise ValueError('Se encontro una etiqueta fuera de LABELS')
    return label_codes


def decode_labels(label_codes):
    return np.asarray(LABELS, dtype=object)[np.asarray(label_codes, dtype=int)]


def optimize_model(X_train, y_train):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = GridSearchCV(
        estimator=build_model(),
        param_grid=XGB_PARAM_GRID,
        scoring='balanced_accuracy',
        cv=inner_cv,
        refit=True,
        n_jobs=-1,
    )
    search.fit(X_train, encode_labels(y_train))
    return search

## Experimento A

Features utilizadas: todas las columnas `_z` disponibles, que corresponden a features derivadas de HR/R-R y ECG.

La matriz de confusion utiliza siempre el orden `Low`, `Medium`, `High`.

In [ ]:
fold_results = []
confusion_matrices = {}

for test_worker in sorted(data):
    train_workers = [worker for worker in sorted(data) if worker != test_worker]
    train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
    test_frame = data[test_worker].copy()

    train_target = train_frame['FatigueLevel']
    test_target = test_frame['FatigueLevel']

    train_mask = train_target.notna()
    test_mask = test_target.notna()
    X_train = train_frame.loc[train_mask, feature_columns]
    y_train = train_target.loc[train_mask]
    X_test = test_frame.loc[test_mask, feature_columns]
    y_test = test_target.loc[test_mask]

    if y_train.nunique() < 2:
        raise ValueError(f'{test_worker}: TRAIN tiene menos de dos clases')
    if y_test.empty:
        raise ValueError(f'{test_worker}: TEST no contiene FatigueIndex valido')

    search = optimize_model(X_train, y_train)
    y_pred = decode_labels(search.predict(X_test))

    confusion_matrices[test_worker] = confusion_matrix(
        y_test, y_pred, labels=LABELS
    )
    matrix = confusion_matrices[test_worker]
    class_recall = np.divide(
        np.diag(matrix),
        matrix.sum(axis=1),
        out=np.zeros(len(LABELS), dtype=float),
        where=matrix.sum(axis=1) != 0,
    )
    fold_results.append({
        'Test_worker': test_worker,
        'Train_workers': ', '.join(train_workers),
        'P33_test_worker': test_frame['P33_worker'].iloc[0],
        'P66_test_worker': test_frame['P66_worker'].iloc[0],
        'N_train': len(y_train),
        'N_test': len(y_test),
        'Best_Params': str(search.best_params_),
        'Recall_Low': class_recall[0],
        'Recall_Medium': class_recall[1],
        'Recall_High': class_recall[2],
        'Accuracy': accuracy_score(y_test, y_pred),
        'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
        'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0),
    })

fold_results_df = pd.DataFrame(fold_results)
mean_row = {column: np.nan for column in fold_results_df.columns}
mean_row['Test_worker'] = 'Media'
mean_row['Train_workers'] = 'Promedio de los folds'
for metric in ['P33_test_worker', 'P66_test_worker', 'N_train', 'N_test', 'Recall_Low', 'Recall_Medium', 'Recall_High', 'Accuracy', 'Balanced_Accuracy', 'Macro_F1']:
    mean_row[metric] = fold_results_df[metric].mean()
fold_results_with_mean = pd.concat(
    [fold_results_df, pd.DataFrame([mean_row])], ignore_index=True
)
fold_results_with_mean

In [ ]:
print('Matrices de confusion por trabajador TEST:')
for worker, matrix in confusion_matrices.items():
    print(f'\nTEST = {worker}')
    print(pd.DataFrame(matrix, index=LABELS, columns=LABELS))

metrics = ['Accuracy', 'Balanced_Accuracy', 'Macro_F1']
summary_df = pd.DataFrame({
    'Metric': metrics,
    'Mean': [fold_results_df[metric].mean() for metric in metrics],
    'Std': [fold_results_df[metric].std(ddof=1) for metric in metrics],
})
print('Media y desviacion estandar de las metricas:')
display(summary_df)

aggregate_confusion = np.sum(
    np.stack([confusion_matrices[worker] for worker in sorted(confusion_matrices)]),
    axis=0,
)
print('Matriz de confusion agregada de los folds:')
display(pd.DataFrame(aggregate_confusion, index=LABELS, columns=LABELS))

class_support = aggregate_confusion.sum(axis=1)
predicted_support = aggregate_confusion.sum(axis=0)
class_metrics_df = pd.DataFrame({
    'Support': class_support,
    'Recall': np.divide(
        np.diag(aggregate_confusion),
        class_support,
        out=np.zeros(len(LABELS), dtype=float),
        where=class_support != 0,
    ),
    'Precision': np.divide(
        np.diag(aggregate_confusion),
        predicted_support,
        out=np.zeros(len(LABELS), dtype=float),
        where=predicted_support != 0,
    ),
}, index=LABELS)
print('Metricas agregadas por clase:')
display(class_metrics_df.round(3))

distribution_rows = []
for worker in sorted(data):
    labels = data[worker]['FatigueLevel']
    counts = labels.value_counts().reindex(LABELS, fill_value=0)
    distribution_rows.append({
        'Test_worker': worker,
        'P33_worker': data[worker]['P33_worker'].iloc[0],
        'P66_worker': data[worker]['P66_worker'].iloc[0],
        'Low': int(counts['Low']),
        'Medium': int(counts['Medium']),
        'High': int(counts['High']),
        'Total_valid': int(counts.sum()),
    })
print('Distribucion individual de clases en cada trabajador:')
display(pd.DataFrame(distribution_rows))

## Experimento C: ablacion ECG

Se conservan solo las features calculadas a partir del ECG. Se excluyen las features HR/R-R y las temporales, porque estas ultimas dependen de HR/R-R. `ECG_missing_peaks` se conserva como feature de calidad de picos ECG. El target sigue siendo `FatigueLevel`, creado individualmente para cada trabajador con sus propios P33 y P66.

In [ ]:
# Reentrenamiento final con todos los trabajadores y optimizacion interna de XGBoost
all_frame = pd.concat([data[worker] for worker in sorted(data)], ignore_index=True)
all_fatigue = pd.to_numeric(all_frame['FatigueIndex'], errors='coerce')
all_target = all_frame['FatigueLevel']
all_mask = all_target.notna()
X_all = all_frame.loc[all_mask, feature_columns]
y_all = all_target.loc[all_mask]

final_search = optimize_model(X_all, y_all)
final_model = final_search.best_estimator_

models_dir = ROOT_DIR / 'models' / 'individual' / 'XGBOOST'
models_dir.mkdir(parents=True, exist_ok=True)
joblib_path = models_dir / 'final_xgboost_individual.joblib'
onnx_path = models_dir / 'final_xgboost_individual.onnx'
metadata_path = models_dir / 'final_xgboost_individual_metadata.json'

joblib.dump(final_model, joblib_path)
onnx_model = convert_xgboost(
    final_model.named_steps['classifier'],
    initial_types=[('features', OnnxFloatTensorType([None, len(feature_columns)]))],
    target_opset=15,
)
onnx_path.write_bytes(onnx_model.SerializeToString())

metadata = {
    'model_type': 'XGBClassifier',
    'validation': 'LOSO_with_individual_worker_labels_and_inner_XGBoost_search',
    'training_workers': sorted(data),
    'feature_columns': feature_columns,
    'feature_count': len(feature_columns),
    'labels': LABELS,
    'preprocessing': 'SimpleImputer(strategy=median), applied before ONNX inference',
    'xgb_param_grid': XGB_PARAM_GRID,
    'selected_params_final': {
        key.replace('classifier__', ''): value
        for key, value in final_search.best_params_.items()
    },
    'target_definition': 'P33/P66 calculados individualmente para cada trabajador',
    'target_column': 'FatigueLevel derived from FatigueIndex',
    'worker_percentiles': {
        worker: {
            'p33': float(data[worker]['P33_worker'].iloc[0]),
            'p66': float(data[worker]['P66_worker'].iloc[0]),
        }
        for worker in sorted(data)
    },
    'temporal_features': [
        'HR_mean_ultimos_5min_z',
        'cambio_HR_vs_baseline_z',
        'tendencia_HR_z',
        'cambio_RMSSD_z',
    ],
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'Modelo final entrenado con {len(y_all)} ventanas y {len(feature_columns)} features')
print(f'Parametros finales seleccionados: {final_search.best_params_}')
print(f'Modelo joblib guardado en: {joblib_path}')
print(f'Modelo ONNX guardado en: {onnx_path}')
print(f'Metadatos guardados en: {metadata_path}')

## Experimento B: ablacion HR/R-R

Se conservan solo las features calculadas a partir de HR/R-R y las cuatro features temporales. Se eliminan las features ECG. El target sigue siendo `FatigueLevel`, creado individualmente para cada trabajador con sus propios P33 y P66.

In [ ]:
HR_RR_FEATURES = [
    'SDNN_z',
    'RMSSD_z',
    'pNN50_z',
    'HR_mean_z',
    'HR_max_z',
    'HR_min_z',
    'HR_mean_ultimos_5min_z',
    'cambio_HR_vs_baseline_z',
    'tendencia_HR_z',
    'cambio_RMSSD_z',
]

ECG_FEATURES = [
    'ECG_energy_z',
    'ECG_mean_z',
    'ECG_missing_peaks_z',
    'ECG_range_z',
    'ECG_samp_ent_z',
    'ECG_std_z',
]


def run_ablation_experiment(experiment_name, selected_features):
    results = []
    for test_worker in sorted(data):
        train_workers = [worker for worker in sorted(data) if worker != test_worker]
        train_frame = pd.concat([data[worker] for worker in train_workers], ignore_index=True)
        test_frame = data[test_worker].copy()
        train_target = train_frame['FatigueLevel']
        test_target = test_frame['FatigueLevel']
        train_mask = train_target.notna()
        test_mask = test_target.notna()
        X_train = train_frame.loc[train_mask, selected_features]
        y_train = train_target.loc[train_mask]
        X_test = test_frame.loc[test_mask, selected_features]
        y_test = test_target.loc[test_mask]
        search = optimize_model(X_train, y_train)
        y_pred = decode_labels(search.predict(X_test))
        results.append({
            'Experiment': experiment_name,
            'Test_worker': test_worker,
            'P33_test_worker': test_frame['P33_worker'].iloc[0],
            'P66_test_worker': test_frame['P66_worker'].iloc[0],
            'N_train': len(y_train),
            'N_test': len(y_test),
            'Feature_count': len(selected_features),
            'Best_Params': str(search.best_params_),
            'Accuracy': accuracy_score(y_test, y_pred),
            'Balanced_Accuracy': balanced_accuracy_score(y_test, y_pred),
            'Macro_F1': f1_score(y_test, y_pred, labels=LABELS, average='macro', zero_division=0),
        })
    return pd.DataFrame(results)


def add_mean_row(results_frame):
    mean_row = {column: np.nan for column in results_frame.columns}
    mean_row['Test_worker'] = 'Media'
    numeric_columns = results_frame.select_dtypes(include='number').columns
    for column in numeric_columns:
        mean_row[column] = results_frame[column].mean()
    return pd.concat([results_frame, pd.DataFrame([mean_row])], ignore_index=True)


hr_rr_results = run_ablation_experiment('Solo HR/RR + temporales', HR_RR_FEATURES)
hr_rr_results_with_mean = add_mean_row(hr_rr_results)
print('Resultados del Experimento B por trabajador y promedio:')
display(hr_rr_results_with_mean.round(4))

## Modelo final y exportacion

La evaluacion LOSO mide la generalizacion dejando un trabajador fuera en cada fold. Despues, el modelo final se reentrena con todos los trabajadores usando sus etiquetas individuales. El pipeline se guarda en `joblib`, ONNX y JSON con los percentiles individuales utilizados.

In [ ]:
ecg_results = run_ablation_experiment('Solo ECG', ECG_FEATURES)
ecg_results_with_mean = add_mean_row(ecg_results)
print('Resultados del Experimento C por trabajador y promedio:')
display(ecg_results_with_mean.round(4))

ec_results = pd.concat([hr_rr_results, ecg_results], ignore_index=True)
ablation_summary = (
    ec_results.groupby('Experiment')[['Accuracy', 'Balanced_Accuracy', 'Macro_F1']]
    .agg(['mean', 'std'])
    .round(4)
)
print('Resumen de las ablaciones:')
display(ablation_summary)

xgboost_summary = pd.DataFrame({
    'Experiment': [
        'A: ECG + HR/RR + temporales',
        'B: HR/RR + temporales',
        'C: ECG',
    ],
    'Feature_count': [
        len(feature_columns),
        len(HR_RR_FEATURES),
        len(ECG_FEATURES),
    ],
    'Accuracy_mean': [
        fold_results_df['Accuracy'].mean(),
        hr_rr_results['Accuracy'].mean(),
        ecg_results['Accuracy'].mean(),
    ],
    'Balanced_Accuracy_mean': [
        fold_results_df['Balanced_Accuracy'].mean(),
        hr_rr_results['Balanced_Accuracy'].mean(),
        ecg_results['Balanced_Accuracy'].mean(),
    ],
    'Macro_F1_mean': [
        fold_results_df['Macro_F1'].mean(),
        hr_rr_results['Macro_F1'].mean(),
        ecg_results['Macro_F1'].mean(),
    ],
})
print('Resumen comparativo de los experimentos XGBoost:')
display(xgboost_summary.round(4))

In [ ]:
import matplotlib.pyplot as plt


def fit_final_feature_model(selected_features):
    X_selected = all_frame.loc[all_mask, selected_features]
    return optimize_model(X_selected, y_all).best_estimator_


models_by_experiment = {
    'A: ECG + HR/RR': (feature_columns, fit_final_feature_model(feature_columns)),
    'B: HR/RR + temporales': (HR_RR_FEATURES, fit_final_feature_model(HR_RR_FEATURES)),
    'C: ECG': (ECG_FEATURES, fit_final_feature_model(ECG_FEATURES)),
}

importance_tables = {}
for experiment_name, (selected_features, model) in models_by_experiment.items():
    importance = model.named_steps['classifier'].feature_importances_
    importance_tables[experiment_name] = (
        pd.DataFrame({'Feature': selected_features, 'Importance': importance})
        .sort_values('Importance', ascending=True)
    )

fig, axes = plt.subplots(1, 3, figsize=(18, 7), constrained_layout=True)
for axis, (experiment_name, importance_table) in zip(axes, importance_tables.items()):
    axis.barh(importance_table['Feature'], importance_table['Importance'], color='#2f6f8f')
    axis.set_title(experiment_name)
    axis.set_xlabel('Importancia XGBoost')
    axis.grid(axis='x', alpha=0.25)

fig.suptitle('Feature importance de XGBoost en los experimentos A, B y C')
plt.show()

print('Features ordenadas por importancia:')
for experiment_name, importance_table in importance_tables.items():
    print(f'\n{experiment_name}')
    display(importance_table.sort_values('Importance', ascending=False).round(4))